# MDP Example 11

## Description

We want to implement the model of MDP proposed by A. Geron in the book  
*Hands-On Machine Learning With Scikit-Learn and Tensorflow: Concepts, Tools, and Techniques to Build Intelligent Systems* (2017).

The MDP is given (Chapter 16 fig 16.8) by:

<img src="../_images/geron.png" alt="picture of the MDP" width="666">

As it can be seen, the number of actions is different in each state. We illustrate here the manner that gives matrices of same dimension for all the action.

This done by adding an action in a state:  
the transition associated to this action jumps in the same state  
the action is roughly penalized

For example the action a2 should be added is s2:  
We add a transition from s2 to s2 in the matrix associated with the action a2 (later matrix P2)  
We penalize the entry associated to state s2 action a2 (coordinate (1,2) in the matrix Reward).

When the  discount factor is 0.95

- the optimal policy is [ 0, 2, 1 ]
- the value function Value is [ 21.8992 1.17982 53.8735 ]

If you change the discount factor in 0.9  then

- the optimal policy should be [0,0,2]
- the value function should be [18.9189  0.0 50.1337]

## Tasks performed

1. Create an MDP

- create two `MarmoteInterval` objects to hold the state Space and the action space
- create a `vector<TransitionStructure*>` a vector of transitionStructure which is the upperclass of `SparseMatrix` to store the transition matrices
- create three `SparseMatrix` objects to hold the transition matrices associated with each of the two actions. They are defined entry by entry with the `setEntry()` function;
- create the `DiscountedMDP`

2. solve the MDP with methods `ValueIteration`, `PolicyIterationModified` and `GaussSeidelValueIteration`

3. Check the obtained costs

4. Clean up.

In [ ]:
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteCore")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteMDP")
#pragma cling add_library_path("/home/assia/miniconda3/envs/xeus-cpp-env/lib")
#pragma cling load("libmarmoteCore.so")
#pragma cling load("libmarmoteMDP.so")


## Code

### General setup

In [ ]:
#include <marmoteCore/marmoteSparseMatrix.h>
#include <marmoteCore/marmoteInterval.h>
#include <marmoteMDP/marmoteDiscountedMDP.h>
#include <marmoteMDP/marmoteFeedbackSolutionMDP.h>
#include <marmoteMDP/marmoteSolutionMDP.h>
#include <marmoteCore/marmotePolicy.h>
#include <marmoteLog/marmoteLog.h>
#include <list>
#include <vector>
#include <string>
using namespace std;

int i;
double beta = 0.95;
string critere("max");
double epsilon = 0.000001;
int maxIter = 700;
double penalty = -100000;
marmote::log::initialize();
marmoteLogI << "I give some informations to the log";
int min = 0;
int max = 2;
int dim_SS = (max-min+1);
MarmoteSet *actionSpace = new MarmoteInterval(0,2);
MarmoteSet *stateSpace = new MarmoteInterval(min,max);
vector<TransitionStructure*> trans(actionSpace->Cardinal());

We only illustrate here how we manage the creation of virtual event toward the same state.

### Transition matrices

In [ ]:
SparseMatrix *P0 = new SparseMatrix(dim_SS); 
/* matrix for the a0 action*/
P0->setEntry(0,0,0.7);
P0->setEntry(0,1,0.3);
P0->setEntry(1,1,1.0);
P0->setEntry(2,2,1.0); /* add virtual action */
trans.at(0) = P0;

/* matrix for the a1 action*/
SparseMatrix *P1 = new SparseMatrix(dim_SS);
P1->setEntry(0,0,1.0);
P1->setEntry(1,2,1.0);
P1->setEntry(2,2,1.0); /* add virtual action */
trans.at(1) = P1;

/* matrix for the a2 action*/
SparseMatrix *P2 = new SparseMatrix(dim_SS);
P2->setEntry(0,0,0.8);
P2->setEntry(0,1,0.2);
P2->setEntry(1,1,1.0); /* add virtual action */
P2->setEntry(2,0,0.8);
P2->setEntry(2,1,0.1);
P2->setEntry(2,2,0.1);
trans.at(2) = P2;

We also underline  that the transition with probability 0 should not be filled in. Indeed, the are not necessarily and take memory and computation time.  
`SparseMatrix` object manages this.

This is the Reward matrix

In [ ]:
SparseMatrix *Reward  = new SparseMatrix(dim_SS);
Reward->setEntry(0,0,7);
Reward->setEntry(0,1,0.0); /* this must not done */
Reward->setEntry(0,2,0.0); /* indeed null entries do not to have been filled in  */
Reward->setEntry(1,0,0);   /* this is inefficient (supplementary computations are done) but this has no consequences */
Reward->setEntry(1,1,-50);
Reward->setEntry(1,2,penalty);
Reward->setEntry(2,0,penalty);
Reward->setEntry(2,1,penalty);
Reward->setEntry(2,2,32);

### Build the reward structures actually used in the C++ example

In [ ]:
vector<TransitionStructure*> rews(actionSpace->Cardinal());
SparseMatrix *R1 = new SparseMatrix(dim_SS);
SparseMatrix *R2 = new SparseMatrix(dim_SS);
SparseMatrix *R3  = new SparseMatrix(dim_SS);

R1->setEntry(0,0,10);
R1->setEntry(2,2,penalty);

R2->setEntry(1,2,-50);
R2->setEntry(2,2,penalty);

R3->setEntry(1,1,penalty);
R3->setEntry(2,0,40);

rews.at(0) = R1;
rews.at(1) = R2;
rews.at(2) = R3;

### Build and print the discounted MDP

In [ ]:
std::cout << "Size :\t" << std::endl ;
std::cout << critere.size() << std::endl;
std::cout<< "Begining MDP building" <<  std::endl ;
DiscountedMDP *mdp1 = new DiscountedMDP(critere, stateSpace, actionSpace, trans, rews,beta);
std::cout<<"End of building MDP" << std::endl ;

std::cout<<"printing MDP" << std::endl ;
mdp1->Write();

### Value iteration and solution checking

In [ ]:
std::cout<<"Printing solution from value iteration" << std::endl ;
FeedbackSolutionMDP *optimum = mdp1->ValueIteration(epsilon, maxIter);
optimum->Write();

cout << endl << "Checking solutions" << endl; 
mdp1->PolicyCost(optimum,epsilon, maxIter);
for(i=0;i<stateSpace->Cardinal();i++){
	cout << "i= " << i << " sol= " << optimum->getValueIndex(i) << endl;
}

### Other solving methods

In [ ]:
cout << endl << "Modified Policy iteration"<< std::endl ;
SolutionMDP *optimum2 = mdp1->PolicyIterationModified(epsilon, maxIter, epsilon*0.01, 100);
optimum2->Write();

cout << endl << "Gauss Seidel Value Iteration"<< std::endl ;
SolutionMDP *optimum3 = mdp1->ValueIterationGS(epsilon, maxIter);
optimum3->Write();

cout << endl << "Modified Policy iteration GS"<< std::endl ;
SolutionMDP *optimum4 = mdp1->PolicyIterationModifiedGS(epsilon, maxIter, 0.001, 20);
optimum4->Write();

### Clean up

In [ ]:
std::cout << std::endl <<"********************************\n" << std::endl ;
cout << "Destructing" << endl;

delete mdp1;
delete optimum;
delete optimum2;
delete optimum3;
delete optimum4;

std::cout<<"Destructing 2" << std::endl ;
delete stateSpace;
delete actionSpace;

## Output

## Download

The source file is here